# Graph Sparsification Experiments

Systematic comparison of **Metric Backbone (MBB)**, **Metric Backbone Rescaled (MBBr)**,
and **Effective Resistance** sparsification across configuration models with various
degree and weight distributions.

**Weight conventions:**
- Generators produce **distance** weights (costs in (0, ∞])
- MBB is computed on **distances**
- MBBr rescales the MBB proximities so their sum matches the original graph
- EffR and SIR use **proximity** weights (in [0, 1]), converted via `proximity = 1/(distance+1)`

Beta is auto-calibrated for each graph so that the SIR epidemic is in an interesting regime.

In [ ]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
src_path = os.path.join(repo_root, 'src', 'python')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

In [ ]:
import numpy as np
from scipy import sparse
import matplotlib.pyplot as plt
import time

from graph_sparsification.generators import configuration_model
from graph_sparsification.sparsifiers import (
    metric_backbone,
    metric_backbone_rescaled,
    effective_resistance_sparsify,
    to_proximity,
    to_distance,
)
from graph_sparsification.sir import sir_monte_carlo, calibrate_beta
from graph_sparsification.visualization import (
    plot_adjacency_comparison,
    plot_multi_infection_comparison,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

try:
    from graph_sparsification._sir_cpp import sir_simulation_cpp
    print('C++ SIR backend: available')
except ImportError:
    print('C++ SIR backend: NOT available (using Python fallback)')

## 1. Distribution definitions

In [ ]:
# ── Degree distributions ──────────────────────────────────────────────

degree_distributions = {
    'Unif(1,50)':  lambda n, rng: rng.integers(1, 51, size=n),
    'Unif(1,100)': lambda n, rng: rng.integers(1, 101, size=n),
    'Exp(30)':     lambda n, rng: np.ceil(rng.exponential(30, size=n)).astype(int),
    'Exp(60)':     lambda n, rng: np.ceil(rng.exponential(60, size=n)).astype(int),
    'LogN(3.26,0.66)': lambda n, rng: np.ceil(rng.lognormal(3.26, 0.66, size=n)).astype(int),
    'LogN(3.26,2)':    lambda n, rng: np.ceil(rng.lognormal(3.26, 2.0, size=n)).astype(int),
    'Pareto(2.5,20)':  lambda n, rng: np.ceil((rng.pareto(2.5, size=n) + 1) * 20).astype(int),
    'Pareto(1.5,30)':  lambda n, rng: np.ceil((rng.pareto(1.5, size=n) + 1) * 30).astype(int),
}

# ── Weight (distance/cost) distributions ──────────────────────────────

weight_distributions = {
    'Exp(1/30)':       lambda m, rng: rng.exponential(30.0, size=m),   # scale = 1/rate
    'Exp(1)':          lambda m, rng: rng.exponential(1.0, size=m),
    'Exp(30)':         lambda m, rng: rng.exponential(1/30, size=m),
    'LogN(2,1)':       lambda m, rng: rng.lognormal(2.0, 1.0, size=m),
    'LogLogN(1.2,0.4)': lambda m, rng: np.exp(rng.lognormal(1.2, 0.4, size=m)),
    'LogLogN(1.2,0.8)': lambda m, rng: np.exp(rng.lognormal(1.2, 0.8, size=m)),
    'LogLogN(2,0.8)':   lambda m, rng: np.exp(rng.lognormal(2.0, 0.8, size=m)),
}

print(f'{len(degree_distributions)} degree distributions × {len(weight_distributions)} weight distributions')
print(f'= {len(degree_distributions) * len(weight_distributions)} configurations')

## 2. Experiment parameters

In [ ]:
N_NODES = 200          # nodes per graph
N_SIR_RUNS = 200       # Monte Carlo SIR runs per graph
N_CAL_RUNS = 30        # calibration runs (fewer, for speed)
GAMMA = 1.0            # recovery rate
SEED = 42

## 3. Main loop

In [ ]:
all_results = []
master_rng = np.random.default_rng(SEED)

for deg_name, deg_sampler in degree_distributions.items():
    for wt_name, wt_sampler in weight_distributions.items():
        label = f'{deg_name}  |  {wt_name}'
        run_seed = int(master_rng.integers(0, 2**31))
        rng = np.random.default_rng(run_seed)

        # ── Generate graph (distance weights) ─────────────────────────
        W_dist = configuration_model(N_NODES, deg_sampler, wt_sampler, rng=rng)
        n_edges = sparse.triu(W_dist).nnz

        if n_edges < 10:
            print(f'SKIP {label}: only {n_edges} edges')
            continue

        degrees = np.diff(W_dist.indptr)
        dist_weights = W_dist.data

        # Convert to proximity for EffR and SIR
        W_prox = to_proximity(W_dist)

        print(f'\n{"="*72}')
        print(f'{label}')
        print(f'  edges={n_edges}, '
              f'deg: mean={degrees.mean():.1f} med={np.median(degrees):.0f} max={degrees.max()}, '
              f'dist: mean={dist_weights.mean():.2f} med={np.median(dist_weights):.2f}, '
              f'prox: mean={W_prox.data.mean():.4f}')

        # ── Calibrate beta (on proximity graph) ──────────────────────
        beta, cal_info = calibrate_beta(
            W_prox, gamma=GAMMA,
            n_calibration_runs=N_CAL_RUNS, rng=rng, verbose=True,
        )

        # ── Sparsify ──────────────────────────────────────────────────

        # MBB (on distances)
        t0 = time.time()
        W_mbb_dist = metric_backbone(W_dist)
        t_mbb = time.time() - t0
        n_mbb = sparse.triu(W_mbb_dist).nnz
        W_mbb_prox = to_proximity(W_mbb_dist)  # convert for SIR

        # MBBr (MBB rescaled — already returns proximity weights)
        t0 = time.time()
        W_mbbr_prox = metric_backbone_rescaled(W_dist)
        t_mbbr = time.time() - t0
        n_mbbr = sparse.triu(W_mbbr_prox).nnz  # same sparsity as MBB

        # MBBr rescaling factor
        mbbr_scale = W_prox.data.sum() / W_mbb_prox.data.sum() if W_mbb_prox.data.sum() > 0 else np.nan

        # EffR (on proximities, exact same edge count as MBB)
        t0 = time.time()
        W_effr_prox = effective_resistance_sparsify(W_prox, n_edges=n_mbb, rng=rng)
        t_effr = time.time() - t0
        n_effr = sparse.triu(W_effr_prox).nnz

        print(f'  MBB:  {n_mbb} edges ({n_mbb/n_edges*100:.1f}%) [{t_mbb:.1f}s]')
        print(f'  MBBr: {n_mbbr} edges ({n_mbbr/n_edges*100:.1f}%) [{t_mbbr:.1f}s]  '
              f'(scale={mbbr_scale:.2f}x)')
        print(f'  EffR: {n_effr} edges ({n_effr/n_edges*100:.1f}%) [{t_effr:.1f}s]')

        # ── SIR Monte Carlo (all on proximity graphs) ─────────────────
        initial = [int(np.argmax(np.array(W_prox.sum(axis=1)).ravel()))]

        sir_orig = sir_monte_carlo(W_prox,      beta, GAMMA, initial, n_runs=N_SIR_RUNS, rng=rng)
        sir_mbb  = sir_monte_carlo(W_mbb_prox,  beta, GAMMA, initial, n_runs=N_SIR_RUNS, rng=rng)
        sir_mbbr = sir_monte_carlo(W_mbbr_prox, beta, GAMMA, initial, n_runs=N_SIR_RUNS, rng=rng)
        sir_effr = sir_monte_carlo(W_effr_prox, beta, GAMMA, initial, n_runs=N_SIR_RUNS, rng=rng)

        # ── Plots ─────────────────────────────────────────────────────
        fig = plot_adjacency_comparison(W_dist, W_mbb_dist,
                                        labels=('Original (dist)', 'Metric Backbone (dist)'))
        fig.suptitle(label, fontsize=13, y=1.02)
        plt.show()

        fig = plot_multi_infection_comparison(
            sir_orig['infection_prob'],
            [sir_mbb['infection_prob'], sir_mbbr['infection_prob'], sir_effr['infection_prob']],
            ['MBB', 'MBBr', 'Eff. Resistance'],
        )
        fig.suptitle(f'{label}  (beta={beta:.4f})', fontsize=13, y=1.02)
        plt.show()

        # ── Store result ──────────────────────────────────────────────
        def _mse(po, ps):
            m = np.isfinite(po) & np.isfinite(ps)
            return np.mean((po[m] - ps[m])**2)

        def _r2(po, ps):
            m = np.isfinite(po) & np.isfinite(ps)
            po, ps = po[m], ps[m]
            ss_tot = np.sum((po - po.mean())**2)
            return 1 - np.sum((po - ps)**2) / ss_tot if ss_tot > 0 else 0.0

        all_results.append({
            'deg': deg_name, 'wt': wt_name, 'label': label,
            'n_edges': n_edges, 'beta': beta,
            'mbb_pct': n_mbb / n_edges * 100,
            'mbbr_pct': n_mbbr / n_edges * 100,
            'effr_pct': n_effr / n_edges * 100,
            'mbbr_scale': mbbr_scale,
            'mse_mbb':  _mse(sir_orig['infection_prob'], sir_mbb['infection_prob']),
            'mse_mbbr': _mse(sir_orig['infection_prob'], sir_mbbr['infection_prob']),
            'mse_effr': _mse(sir_orig['infection_prob'], sir_effr['infection_prob']),
            'r2_mbb':  _r2(sir_orig['infection_prob'], sir_mbb['infection_prob']),
            'r2_mbbr': _r2(sir_orig['infection_prob'], sir_mbbr['infection_prob']),
            'r2_effr': _r2(sir_orig['infection_prob'], sir_effr['infection_prob']),
            'mean_inf_orig': sir_orig['infection_prob'].mean(),
        })

## 4. Summary table

In [ ]:
print(f"{'Degree':<18} {'Weights':<18} {'edges':>6} {'beta':>8} "
      f"{'MBB%':>6} {'MBBr%':>6} {'EffR%':>6} {'R² MBB':>7} {'R² MBBr':>8} {'R² EffR':>8}")
print('-' * 110)
for r in all_results:
    print(f"{r['deg']:<18} {r['wt']:<18} {r['n_edges']:>6} {r['beta']:>8.4f} "
          f"{r['mbb_pct']:>5.1f}% {r['mbbr_pct']:>5.1f}% {r['effr_pct']:>5.1f}% "
          f"{r['r2_mbb']:>7.3f} {r['r2_mbbr']:>8.3f} {r['r2_effr']:>8.3f}")

## 5. Aggregate comparison plot

In [ ]:
labels = [r['label'] for r in all_results]
r2_mbb  = [r['r2_mbb']  for r in all_results]
r2_mbbr = [r['r2_mbbr'] for r in all_results]
r2_effr = [r['r2_effr'] for r in all_results]

y = np.arange(len(labels))
h = 0.25

fig, ax = plt.subplots(figsize=(10, max(5, len(labels) * 0.5)))
ax.barh(y + h,   r2_mbb,  h, label='Metric Backbone',         color='steelblue')
ax.barh(y,       r2_mbbr, h, label='MBB Rescaled',             color='mediumseagreen')
ax.barh(y - h,   r2_effr, h, label='Effective Resistance',     color='coral')
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel('$R^2$ (infection probability)', fontsize=12)
ax.set_title('Sparsifier fidelity across degree × weight distributions', fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(0, 1.05)
ax.axvline(1.0, color='gray', ls='--', alpha=0.3)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Heatmaps by configuration

Per-configuration heatmaps for: MBBr rescaling factor, percentage of edges preserved, and MSE of infection probabilities for each method.

In [ ]:
deg_names = list(degree_distributions.keys())
wt_names  = list(weight_distributions.keys())
n_deg, n_wt = len(deg_names), len(wt_names)

# Build lookup from results
res_lookup = {(r['deg'], r['wt']): r for r in all_results}

def _build_grid(key):
    grid = np.full((n_deg, n_wt), np.nan)
    for i, d in enumerate(deg_names):
        for j, w in enumerate(wt_names):
            if (d, w) in res_lookup:
                grid[i, j] = res_lookup[(d, w)][key]
    return grid

def _plot_heatmap(grid, title, cmap='viridis', fmt='.2f', vmin=None, vmax=None):
    fig, ax = plt.subplots(figsize=(max(8, n_wt * 1.1), max(4, n_deg * 0.7)))
    im = ax.imshow(grid, cmap=cmap, aspect='auto', vmin=vmin, vmax=vmax)
    ax.set_xticks(range(n_wt))
    ax.set_xticklabels(wt_names, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(n_deg))
    ax.set_yticklabels(deg_names, fontsize=9)
    ax.set_xlabel('Weight distribution', fontsize=11)
    ax.set_ylabel('Degree distribution', fontsize=11)
    ax.set_title(title, fontsize=13, pad=10)
    # Annotate cells
    for i in range(n_deg):
        for j in range(n_wt):
            val = grid[i, j]
            if np.isfinite(val):
                color = 'white' if im.norm(val) < 0.5 else 'black'
                ax.text(j, i, f'{val:{fmt}}', ha='center', va='center',
                        fontsize=8, color=color)
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    return fig

# ── 1. MBBr rescaling factor ─────────────────────────────────────────
grid_scale = _build_grid('mbbr_scale')
fig = _plot_heatmap(grid_scale, 'MBBr rescaling factor', cmap='RdYlGn_r')
plt.show()

# ── 2. Percentage of edges preserved (same for MBB/MBBr/EffR) ────────
grid_pct = _build_grid('mbb_pct')
fig = _plot_heatmap(grid_pct, 'Edges preserved (%)', cmap='YlOrRd', fmt='.1f',
                    vmin=0, vmax=100)
plt.show()

# ── 3. MSE heatmaps per method ───────────────────────────────────────
grid_mse_mbb  = _build_grid('mse_mbb')
grid_mse_mbbr = _build_grid('mse_mbbr')
grid_mse_effr = _build_grid('mse_effr')

# Shared color scale across the 3 MSE heatmaps
all_mse = np.concatenate([grid_mse_mbb.ravel(), grid_mse_mbbr.ravel(), grid_mse_effr.ravel()])
mse_max = np.nanmax(all_mse)

for grid, name in [(grid_mse_mbb, 'MBB'), (grid_mse_mbbr, 'MBBr'), (grid_mse_effr, 'Eff. Resistance')]:
    fig = _plot_heatmap(grid, f'MSE of infection prob. — {name}',
                        cmap='magma_r', fmt='.4f', vmin=0, vmax=mse_max)
    plt.show()